# Решения: Selection sort, mergesort и идея quicksort

**Для преподавателя.** Полный эталон к `lesson.ipynb` и `homework.ipynb`; ученикам до сдачи не показывать.

In [ ]:
from pathlib import Path
import math
import statistics
import time
import pandas as pd


def find_csv(name):
    for path in (
        Path(name),
        Path("../") / name,
        Path("../../data") / name,
        Path("../data") / name,
        Path("../../../data") / name,
    ):
        if path.exists():
            return path.resolve()
    return "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_07_bank_arrays_search/data/" + name


unsorted_df = pd.read_csv(find_csv("bank_transactions_unsorted.csv"))
by_id_df = pd.read_csv(find_csv("bank_transactions_sorted_by_txn_id.csv"))
by_amount_df = pd.read_csv(find_csv("bank_transactions_sorted_by_amount.csv"))
tiny_df = pd.read_csv(find_csv("bank_transactions_tiny.csv"))
COLS = ["txn_id", "amount", "day", "risk_score"]
unsorted_txns = list(unsorted_df[COLS].itertuples(index=False, name=None))
id_txns = list(by_id_df[COLS].itertuples(index=False, name=None))
amount_txns = list(by_amount_df[COLS].itertuples(index=False, name=None))
tiny_txns = list(tiny_df[COLS].itertuples(index=False, name=None))
id_list = [row[0] for row in id_txns]
amount_list = [row[1] for row in amount_txns]
assert id_list == sorted(id_list)
assert amount_list == sorted(amount_list)
print(f"Загружено {len(unsorted_txns)} транзакций; поля кортежа: {COLS}")


## Урок. 1–4. Selection, merge, mergesort

In [ ]:
def linear_search(values, target):
    for index, value in enumerate(values):
        if value == target:
            return index
    return -1


def binary_search(values, target):
    left, right = 0, len(values) - 1
    while left <= right:
        mid = (left + right) // 2
        if values[mid] == target:
            return mid
        if values[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1


def lower_bound(values, target):
    left, right = 0, len(values)
    while left < right:
        mid = (left + right) // 2
        if values[mid] < target:
            left = mid + 1
        else:
            right = mid
    return left


def upper_bound(values, target):
    left, right = 0, len(values)
    while left < right:
        mid = (left + right) // 2
        if values[mid] <= target:
            left = mid + 1
        else:
            right = mid
    return left


def selection_sort(values):
    result = list(values)
    for i in range(len(result)):
        smallest = i
        for j in range(i + 1, len(result)):
            if result[j] < result[smallest]:
                smallest = j
        result[i], result[smallest] = result[smallest], result[i]
    return result


def merge_sorted(left, right):
    i = j = 0
    result = []
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i]); i += 1
        else:
            result.append(right[j]); j += 1
    return result + list(left[i:]) + list(right[j:])


def merge_sort(values):
    if len(values) <= 1:
        return list(values)
    mid = len(values) // 2
    return merge_sorted(merge_sort(values[:mid]), merge_sort(values[mid:]))


def median_runtime(function, values, repeats=3):
    samples = []
    for _ in range(repeats):
        start = time.perf_counter()
        function(list(values))
        samples.append(time.perf_counter() - start)
    return statistics.median(samples)

values = [9, 1, 5, 3, 7]
smallest_index = min(range(len(values)), key=values.__getitem__)
one_step = values.copy(); one_step[0], one_step[smallest_index] = one_step[smallest_index], one_step[0]
assert one_step == [1, 9, 5, 3, 7]


## Урок. 5. Partition

In [ ]:
def partition(values, pivot):
    return ([x for x in values if x < pivot], [x for x in values if x == pivot], [x for x in values if x > pivot])

less, equal, greater = partition([8, 2, 8, 4, 9, 8], 8)
assert len(less) + len(equal) + len(greater) == 6


## Урок. 6–7. Работа и банковские данные

In [ ]:
def selection_comparisons(n):
    return n * (n - 1) // 2

amounts80 = [r[1] for r in unsorted_txns[:80]]
selection_result = selection_sort(amounts80); merge_result = merge_sort(amounts80)
assert selection_result == merge_result == sorted(amounts80)


## Урок. 8–9. Вывод и gate

In [ ]:
SORT_NOTE = "Selection sort на каждом шаге просматривает остаток и делает O(n²) сравнений. Mergesort строит уровни разбиения и слияния за O(n log n). Quicksort в среднем похож по росту, но при неудачном pivot может деградировать до O(n²)."
source = [4, 1, 4, -2, 0]; selection_sort(source)
checks = {"copy": source == [4, 1, 4, -2, 0], "duplicates": merge_sort([2, 2, 1]).count(2) == 2, "empty": selection_sort([]) == merge_sort([]) == [], "bank_data": selection_result == sorted(amounts80)}
assert set(checks.values()) == {True}


## ДЗ. Part A

In [ ]:
risks = [r[3] for r in unsorted_txns[:120]]; risk_selection = selection_sort(risks)
ids = [r[0] for r in unsorted_txns[:300]]; id_merge = merge_sort(ids)
cases = [[], [1], [2, 1], [3, 3, 1], list(range(20, -1, -1))]
property_checks = [fn(case) == sorted(case) for case in cases for fn in (selection_sort, merge_sort)]
assert all(property_checks)


## ДЗ. Challenge

In [ ]:
def quick_sort(values):
    if len(values) <= 1: return list(values)
    pivot = values[len(values) // 2]
    less, equal, greater = partition(values, pivot)
    return quick_sort(less) + equal + quick_sort(greater)

QUICK_NOTE = "Если pivot каждый раз оказывается минимумом или максимумом, одна часть почти пуста, глубина рекурсии становится n, а суммарная работа — O(n²). Случайный или медианный pivot уменьшает риск, но не отменяет худший случай."
assert quick_sort([3, 1, 3, 2, 3]) == [1, 2, 3, 3, 3] and len(QUICK_NOTE) >= 180
